# Running Gaussian Jobs

Below is an example script for a Gaussian 16 job, which should be submitted from the `/scratch/projects/[project-code]/` storage area.

The script passes your Slurm allocation to Gaussian. It sets `GAUSS_PDEF` (cores) from `--cpus-per-task` and `GAUSS_MDEF` (memory) from the job's memory, minus a 4 GB margin for Gaussian's own overhead. Don't set `%Mem`, `%NProcShared` or `%CPU` in your input files, because these override the script (see [Input files](#input-files-dont-set-mem-nprocshared-or-cpu) below).

Before Gaussian starts, the script prints a short diagnostics block to the job's `.out` file. When Gaussian finishes, it prints the exit code and how much scratch space was used.

```bash

#!/bin/bash

#SBATCH --job-name=g16-test
#SBATCH --account=[your-account]

#SBATCH --partition=chemistry
#SBATCH --output=%j.out
#SBATCH --error=%j.err

# Job time
#SBATCH --time=1-0            # 1 day. Max wall time is 3 days

## Gaussian uses single nodes
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=12    # Cores for Gaussian. Use 48 or less

# Memory: leave unset to use the partition default. Only request more
# if past runs show the calculation needs it, for example:
##SBATCH --mem=64G

module purge
module load gaussian/16/C.01

source $g16profile

# Pass the Slurm allocation to Gaussian
cpus=${SLURM_CPUS_PER_TASK:-$SLURM_CPUS_ON_NODE}
if [[ -n $SLURM_MEM_PER_NODE ]]; then
  total=$SLURM_MEM_PER_NODE
else
  total=$(( SLURM_MEM_PER_CPU * SLURM_CPUS_ON_NODE ))
fi
export GAUSS_PDEF=$cpus
export GAUSS_MDEF="$(( total - 4096 ))MB"   # 4 GB margin for Gaussian's overhead

# Setup scratch directory for Gaussian
export GAUSS_SCRDIR=/tmp/$USER/$SLURM_JOB_ID
mkdir -p $GAUSS_SCRDIR
chmod 700 $GAUSS_SCRDIR

input="inputs/example.gjf"

echo "=== Gaussian job diagnostics ==="
echo "Job ${SLURM_JOB_ID} on $(hostname), partition ${SLURM_JOB_PARTITION}"
echo "CPUs requested: ${cpus}   cores allowed: $(grep Cpus_allowed_list /proc/self/status | cut -f2)"
echo "Slurm memory: ${total} MB   GAUSS_MDEF=${GAUSS_MDEF}   GAUSS_PDEF=${GAUSS_PDEF}"
echo "Scratch: ${GAUSS_SCRDIR} ($(df -h --output=avail "${GAUSS_SCRDIR}" | tail -1) free)"
echo "Gaussian binary: $(command -v g16)"
echo "Link 0 lines in input (these override the above):"
grep -iE '^[[:space:]]*%(mem|nproc|cpu)' "${input}" || echo "  none"
echo "================================"

g16 < "${input}" > output.log

rc=$?
echo "Gaussian exit code: ${rc}"
echo "Scratch used: $(du -sh "${GAUSS_SCRDIR}" | cut -f1)"

# Clean Up
rm -rf $GAUSS_SCRDIR

exit ${rc}

```

Assuming this file is saved as `example.slm`, it can be submitted with the command:
```console
sbatch example.slm
```

## GAUSS_SCRDIR

As part of the prejob commands for slurm, a temporary directory is created in `/tmp/<user_id>/[id]`, where `[id]` is the slurm id of the job. This directory is automatically exported as `$GAUSS_SCRDIR` and should not be altered. This both improves performance of your job by performing write-heavy operations on the local NVMe drives of the compute node, prevents `/scratch` from being slowed down, and cleans up files for you after jobs have completed.

If for some reason you need these files, you have read access to this directory during the job but files will be deleted at the end of the slurm job.

## Best practice: use no more than 48 cores per Gaussian job

The `chemistry` partition nodes have **48 physical cores**, shown to Slurm as **96 CPUs**. A single Gaussian job runs fastest when its thread count matches the physical cores. Running more threads than that makes the job slower, not faster.

Set the number of cores with **`--cpus-per-task` in your Slurm script**, not in the input file. The example scripts pass this value to Gaussian for you.

```bash
#SBATCH --cpus-per-task=48
```

### Benchmark results

Wall times for two production inputs, each run as a single job on a `chemistry` node:

| Threads | Input 1 | Input 2 |
|---|---|---|
| 24 | 12:03:41 | 25:08:49 |
| **48** | **08:22:31** | **18:39:06** |
| 96 | 09:06:39<br>(8% slower than 48) | 22:48:47<br>(22% slower than 48) |

Going from 24 to 48 threads cut the runtime by roughly 25–30%. 96 threads was up to 22% slower than 48, depending on the calculation.

**In summary:** request `--cpus-per-task=48` or less for a single Gaussian job.

## Memory

In most cases you don't need to request memory. The partition default is enough for most Gaussian jobs, so leave `--mem` out of your script.

Request `--mem` only when past runs show the calculation needs more, for example a previous job ran out of memory:

```bash
#SBATCH --mem=64G
```

The example scripts pass the extra memory on to Gaussian automatically, so you don't need to change anything else.

## Input files: don't set %Mem, %NProcShared or %CPU

Don't put `%Mem`, `%NProcShared` or `%CPU` lines in your `.gjf` input files. The job script now sets memory and cores from your Slurm allocation.

- **Input files reused from elsewhere** (another cluster, a colleague, an older project) often contain these lines. Remove them before submitting. They override the script, and a `%Mem` larger than your allocation gets the job killed, often hours into the run.
- **`%CPU`** pins Gaussian to fixed core numbers, which conflicts with Slurm choosing which cores your job runs on.

To find these lines in your input files:

```console
grep -iE '^[[:space:]]*%(mem|nproc|cpu)' *.gjf
```

## Checking your job

Once the job has started, check that Gaussian is using the cores you asked for. Near the top of the Gaussian `.log` file you should see:

```
Will use up to   48 processors via shared memory.
```

The number should match your `--cpus-per-task`. If it doesn't, a `%NProcShared` or `%CPU` line in your input file is overriding the script.

The diagnostics block at the top of the job's `.out` file shows the cores and memory Gaussian was given. Under "Link 0 lines in input" it should say `none`.